# 07. Transparent Baseline Recommendation Model

This notebook builds an understandable inventory-ranking baseline. It is not machine learning yet. Its purpose is to establish a benchmark, reveal missing features, and produce explanations for every score.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

WORKSPACE = Path.cwd()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent

sys.path.insert(0, str(WORKSPACE / "src"))

from venture_ds import (
    load_exports,
    load_optional_export,
    safe_rate,
    wilson_interval,
    ordered_session_funnel,
    add_sample_warning,
)

exports = load_exports(WORKSPACE / "data" / "raw")
events = exports["events"]
sessions = exports["sessions"]
feedback = exports["feedback"]
inventory = exports["inventory"]

exports["_files"]

,dataset,path,found
0,events,C:\Users\ppmal\Documents\Projects\venture-prod...,True
1,sessions,C:\Users\ppmal\Documents\Projects\venture-prod...,True
2,feedback,C:\Users\ppmal\Documents\Projects\venture-prod...,True
3,inventory,C:\Users\ppmal\Documents\Projects\venture-prod...,True


In [2]:
from venture_ds.recommendation import (
    RecommendationRequest,
    transparent_inventory_score,
)

request = RecommendationRequest(
    preferred_categories=("Restaurant", "Activity", "Live Music"),
    preferred_neighbourhoods=("Downtown", "Exchange District", "Osborne Village"),
    include_events=True,
)

if inventory.empty:
    print("No inventory export found.")
else:
    ranked = transparent_inventory_score(inventory, request)
    display(
        ranked[
            [
                "name",
                "entity_type",
                "category",
                "neighbourhood",
                "status",
                "verification_status",
                "score",
                "score_explanation",
            ]
        ].head(25)
    )

,name,entity_type,category,neighbourhood,status,verification_status,score,score_explanation
27,Ai-Kon 2026,event,Anime Convention,Downtown,published,official_source,65.0,published (+25); trusted verification (+25); p...
23,Clementine,place,Brunch,Exchange District,published,official_source,65.0,published (+25); trusted verification (+25); p...
24,Deer + Almond,place,Fine dining,Exchange District,published,official_source,65.0,published (+25); trusted verification (+25); p...
25,RBC Convention Centre Winnipeg,place,Convention Centre,Downtown,published,official_source,65.0,published (+25); trusted verification (+25); p...
22,BMC Market Osborne,place,Mexican,South Osborne,published,official_source,50.0,published (+25); trusted verification (+25);
26,The Forks Market,place,Food Hall & Market,The Forks,published,official_source,50.0,published (+25); trusted verification (+25);
0,12 Resto Bar,place,Bar And Grill Restaurant,NaN,published,unverified,-20.0,published (+25); missing neighbourhood (-15); ...
11,529 Wellington Steakhouse,place,Steakhouse,NaN,published,unverified,-20.0,published (+25); missing neighbourhood (-15); ...
21,A Little Pizza Heaven,place,Pizza Restaurant,NaN,published,unverified,-20.0,published (+25); missing neighbourhood (-15); ...
20,A l'epi de ble - French Bakery,place,Bakery,NaN,published,unverified,-20.0,published (+25); missing neighbourhood (-15); ...


## Why this matters

A transparent baseline gives Venture something measurable to beat. A future learned model should only be adopted when it improves relevant outcomes without harming trust, diversity, or route completion.

## Features still needed for a stronger model

- hours and schedule compatibility
- price and budget fit
- travel time and route cost
- interest-tag overlap
- dietary and accessibility compatibility
- freshness confidence
- actual impressions, selections, saves, replacements, and feedback by listing